# Specialiserede Modeller — 3: CNN — netværk der kan SE

Husker I MNIST-notebookens sidste refleksion? *Netværket ser 784 tal i én lang række og
aner ikke, at pixels har naboer.* I dag indløser vi det løfte: **konvolutionelle neurale
netværk (CNN'er)** er bygget PRÆCIS til at udnytte, at billeder har struktur — og de er
grunden til, at computere i dag kan se.

Dagens data: **FashionMNIST** — 28×28-billeder af tøj i 10 kategorier. Samme format som
MNIST, men markant sværere (prøv selv at skelne en skjorte fra en frakke i 28×28...).

> Denne notebook er selvkørende og kræver kun viden fra Intro-ML — du kan tage emnets notebooks i den rækkefølge, du vil. Der er med vilje flere opgaver, end du kan nå: opgaver mærket **(ekstra)** er til de hurtige, og opgaver mærket **(find fejlen)** indeholder en bevidst fejl, som skal findes og rettes.

## Setup

In [ ]:
!pip install -q kagglehub

In [ ]:
# Plottehjælpere + FashionMNIST fra GitHub (Plan B: upload filerne via mappeikonet i Colab)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Specialiserede-Modeller/hjaelpefunktioner.py
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Specialiserede-Modeller/data/fashion_traen_lille.csv.gz
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Specialiserede-Modeller/data/fashion_test_lille.csv.gz

In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from hjaelpefunktioner import vis_billeder, plot_forvirringsmatrix

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
traen_df = pd.read_csv("fashion_traen_lille.csv.gz")
test_df = pd.read_csv("fashion_test_lille.csv.gz")
print("træning:", traen_df.shape, "| test:", test_df.shape)

> **Plan B:** Hvis Kaggle driller, så fjern `#`'erne nedenfor og kør — filerne fra
> vores GitHub er allerede skåret ned, så spring i så fald `sample`-linjerne ovenfor over.

Formatet er præcis som MNIST i Intro-ML: første kolonne er `label`, resten er 784
pixels. Men nu er der en VIGTIG ny detalje: et CNN vil have billederne som **billeder**,
ikke som flade rækker. Shapen skal være `(antal, kanaler, højde, bredde)` — kanaler er
1 for gråtoner (og 3 for RGB-farver, det møder I i -opgaven til sidst):

In [ ]:
klassenavne = ["T-shirt", "bukser", "sweater", "kjole", "frakke",
               "sandal", "skjorte", "sneaker", "taske", "støvle"]

X_traen = torch.tensor(traen_df.drop(columns=["label"]).values / 255.0,
                       dtype=torch.float32).reshape(-1, 1, 28, 28)
y_traen = torch.tensor(traen_df["label"].values, dtype=torch.long)
X_test = torch.tensor(test_df.drop(columns=["label"]).values / 255.0,
                      dtype=torch.float32).reshape(-1, 1, 28, 28)
y_test = torch.tensor(test_df["label"].values, dtype=torch.long)

print("X_traen:", X_traen.shape, "  ← (antal, kanaler, højde, bredde)")
vis_billeder(X_traen, y_traen, n=10, klassenavne=klassenavne)

# 5: Konvolution — den glidende linse

## Hvorfor ikke bare et dense-netværk?

I MNIST-notebooken fladede vi billedet ud til 784 tal, og hver eneste pixel fik sin
egen vægt. To problemer:

1. **Naboskab smides væk**: for netværket er pixel 5 og pixel 33 bare to vilkårlige
 tal — det aner ikke, at de sidder lige oven over hinanden.
2. **Intet genbrug**: netværket lærer at genkende en snude i øverste venstre hjørne —
 og aner så INTET, når snuden optræder nederst til højre. Alt skal læres forfra for
 hver position.

## Idéen: en lille kerne, der glider

En **konvolution** løser begge dele. Tag en lille matrix — en **kerne** (fx 2×2 eller
3×3) — og lad den GLIDE hen over billedet. På hver position ganges kernen med det
stykke af billedet, den ligger oven på, og summen skrives i output. Samme kerne,
alle positioner:

In [ ]:
billede = torch.tensor([[1., 2., 0., 1.],
                        [0., 1., 3., 1.],
                        [2., 1., 0., 0.],
                        [1., 3., 1., 2.]])

kerne = torch.tensor([[1., 0.],
                      [0., -1.]])

# F.conv2d vil have shapen (antal, kanaler, højde, bredde) — derfor reshape:
resultat = F.conv2d(billede.reshape(1, 1, 4, 4), kerne.reshape(1, 1, 2, 2))
print(resultat.squeeze())

Tjek det øverste venstre tal i hånden: kernen ligger på $\begin{pmatrix}1 & 2\\0 & 1\end{pmatrix}$,
og $1\cdot 1 + 2\cdot 0 + 0\cdot 0 + 1\cdot(-1) = 0$. Kernen `[[1,0],[0,-1]]` beregner
altså "pixel minus dens diagonale nabo" — den reagerer på ÆNDRINGER på skrå.

Bemærk output-størrelsen: et 4×4-billede og en 2×2-kerne giver 3×3 output — kernen kan
stå på 3×3 forskellige positioner. Generelt: **output = billede − kerne + 1** (uden
padding og stride, mere om dem om lidt).

## Kerner kan SE ting — fx kanter

Her kommer det fede: forskellige kerner opdager forskellige mønstre. Den her berømte
kerne (Sobel-kernen) reagerer kraftigt på **lodrette kanter** — steder hvor billedet
skifter fra lyst til mørkt i vandret retning:

In [ ]:
lodret_kant = torch.tensor([[1., 0., -1.],
                            [2., 0., -2.],
                            [1., 0., -1.]])

sko = X_traen[y_traen == 7][0]        # en sneaker fra datasættet

kanter = F.conv2d(sko.reshape(1, 1, 28, 28), lodret_kant.reshape(1, 1, 3, 3))

fig, akser = plt.subplots(1, 2, figsize=(8, 4))
akser[0].imshow(sko.squeeze(), cmap="gray")
akser[0].set_title("original")
akser[1].imshow(kanter.squeeze(), cmap="gray")
akser[1].set_title("lodret-kant-kerne")
for akse in akser:
    akse.axis("off")
plt.show()

Kernen har fremhævet skoens lodrette kanter og ignoreret resten! I et CNN er det
PRÆCIS sådan, de første lag ser: små kerner, der finder kanter, streger og pletter. Den
eneste forskel er, at CNN'et **lærer kernernes tal selv** med gradient descent — i
stedet for at få dem foræret af os.

## Padding, stride og pooling

Tre små begreber, så kan I læse enhver CNN-arkitektur:

- **Padding**: læg en ramme af nuller om billedet, så kernen også kan stå på kanten —
 `padding=1` med en 3×3-kerne bevarer billedstørrelsen.
- **Stride**: lad kernen hoppe flere pixels ad gangen — `stride=2` halverer outputtet.
- **Max-pooling**: skru ned for opløsningen ved at tage MAKSIMUM i små vinduer —
 `MaxPool2d(2)` deler billedet i 2×2-felter og beholder ét tal pr. felt. Det gør
 netværket mindre pixel-pernittengrynet ("der var en kant CIRKA her").

Formlen for output-størrelsen: $\text{ud} = \lfloor(\text{ind} - \text{kerne} + 2\cdot\text{padding})/\text{stride}\rfloor + 1$

In [ ]:
tal = torch.tensor([[1., 3., 2., 4.],
                    [5., 2., 1., 0.],
                    [0., 1., 6., 2.],
                    [3., 2., 1., 1.]])

print(F.max_pool2d(tal.reshape(1, 1, 4, 4), kernel_size=2).squeeze())

### Opgaver

##### Opgave 5.1
Regn (i hånden!) det MIDTERSTE tal i 3×3-outputtet fra det første eksempel (4×4-billedet
og kernen `[[1,0],[0,-1]]`). Kernen ligger så dens øverste venstre hjørne står på
position (række 1, kolonne 1) — altså på tallet 1 i midten. Tjek dit facit mod
`resultat` ovenfor.

*Skriv din udregning her:* $\dots$

##### Opgave 5.2
Udfyld det lille billede og den lille kerne, regn ALLE fire output-tal i hånden — og
tjek jer selv med `F.conv2d`. (3×3-billede, 2×2-kerne → 2×2 output.)

In [ ]:
mit_billede = torch.tensor([[2., 0., 1.],
                            [1., 3., 0.],
                            [0., 2., 2.]])
min_kerne = torch.tensor([[1., -1.],
                          [0., 1.]])

# Regn de fire tal i hånden FØRST — skriv jeres gæt her:
# øverst venstre: ...   øverst højre: ...
# nederst venstre: ...  nederst højre: ...

print(F.conv2d(mit_billede.reshape(1, 1, 3, 3), min_kerne.reshape(1, 1, 2, 2)).squeeze())

##### Opgave 5.3
Prøv de to kerner nedenfor på sneaker-billedet (genbrug kant-cellen). Hvad gør de —
og hvorfor giver det mening ud fra tallene i dem?

In [ ]:
identitet = torch.tensor([[0., 0., 0.],
                          [0., 1., 0.],
                          [0., 0., 0.]])

udtvaering = torch.ones(3, 3) / 9     # alle tal er 1/9

kerne_test = identitet    # ← prøv begge
ud = F.conv2d(sko.reshape(1, 1, 28, 28), kerne_test.reshape(1, 1, 3, 3))
fig, akser = plt.subplots(1, 2, figsize=(8, 4))
akser[0].imshow(sko.squeeze(), cmap="gray")
akser[1].imshow(ud.squeeze(), cmap="gray")
for akse in akser:
    akse.axis("off")
plt.show()

##### Opgave 5.4
Sobel-kernen fandt LODRETTE kanter. Byg dens søster, der finder VANDRETTE kanter (vip
tallene 90 grader — eller brug `.T`, transponering), og kør den på sneakeren og på en
taske (`X_traen[y_traen == 8][0]`). Sammenlign de to kant-billeder.

In [ ]:
vandret_kant = torch.tensor([[..., ..., ...],
                             [..., ..., ...],
                             [..., ..., ...]])

ud = F.conv2d(sko.reshape(1, 1, 28, 28), vandret_kant.reshape(1, 1, 3, 3))
plt.imshow(ud.squeeze(), cmap="gray")
plt.axis("off")
plt.show()

##### Opgave 5.5 (find fejlen)
Koden vil køre en kerne på et billede, men PyTorch protesterer over dimensionerne. Læs
fejlbeskeden og fix koden. (Hvilken shape kræver `F.conv2d` — og hvad har `sko` frisk
fra datasættet?)

In [ ]:
kerne = torch.tensor([[1., -1.]])
ud = F.conv2d(sko.squeeze(), kerne)
print(ud.shape)

##### Opgave 5.6
Kør 3×3-kernen på sneakeren med `padding=0`, `padding=1` og `padding=5` (ekstra
argument til `F.conv2d`). Tjek output-shapen hver gang — og kig på billedet ved
padding=5: hvad er det for en sort ramme?

In [ ]:
ud = F.conv2d(sko.reshape(1, 1, 28, 28), lodret_kant.reshape(1, 1, 3, 3), padding=0)
print(ud.shape)
plt.imshow(ud.squeeze(), cmap="gray")
plt.axis("off")
plt.show()

##### Opgave 5.7
Udfyld formlen $\text{ud} = (\text{ind} - \text{kerne} + 2\cdot\text{padding})/\text{stride} + 1$
som Python-funktion, og tjek den mod PyTorch i de tre tilfælde.

In [ ]:
def output_stoerrelse(ind, kerne, padding, stride):
    return ...

for kerne_str, padding, stride in [(3, 0, 1), (3, 1, 2), (5, 2, 2)]:
    beregnet = output_stoerrelse(28, kerne_str, padding, stride)
    faktisk = F.conv2d(sko.reshape(1, 1, 28, 28),
                       torch.ones(1, 1, kerne_str, kerne_str),
                       padding=padding, stride=stride).shape[-1]
    print(f"kerne {kerne_str}, padding {padding}, stride {stride}: formel {beregnet}, PyTorch {faktisk}")

##### Opgave 5.8
Kør max-pooling (`F.max_pool2d(..., kernel_size=2)`) på kant-billedet fra
Sobel-eksemplet, og derefter ENDNU en gang på resultatet. Kig på de tre billeder —
hvad overlever poolingen, og hvad forsvinder?

In [ ]:
kanter = F.conv2d(sko.reshape(1, 1, 28, 28), lodret_kant.reshape(1, 1, 3, 3))
poolet = F.max_pool2d(kanter, kernel_size=2)
# ...og pool én gang til!

fig, akser = plt.subplots(1, 2, figsize=(8, 4))
akser[0].imshow(kanter.squeeze(), cmap="gray")
akser[0].set_title(f"kanter {tuple(kanter.squeeze().shape)}")
akser[1].imshow(poolet.squeeze(), cmap="gray")
akser[1].set_title(f"poolet {tuple(poolet.squeeze().shape)}")
for akse in akser:
    akse.axis("off")
plt.show()

##### Opgave 5.9
Dense-laget lærer én vægt pr. pixel pr. neuron. Konvolutionslaget genbruger de samme
9 tal (3×3-kernen) over HELE billedet. Giv to grunde til, at genbruget er smart —
tænk på (1) antal parametre og (2) hvad der sker, når motivet flytter sig i billedet.

*Skriv jeres svar her:* $\dots$

##### Opgave 5.10 (ekstra)
Design jeres egen kerne, der reagerer på **diagonale** kanter (fra øverst-venstre mod
nederst-højre). Test den på sneakeren og tasken — og find på mindst ét tjek, der
overbeviser jer om, at den gør det, I tror.

In [ ]:
diagonal_kerne = torch.tensor([[..., ..., ...],
                               [..., ..., ...],
                               [..., ..., ...]])

ud = F.conv2d(sko.reshape(1, 1, 28, 28), diagonal_kerne.reshape(1, 1, 3, 3))
plt.imshow(ud.squeeze(), cmap="gray")
plt.axis("off")
plt.show()

# 6: Byg et CNN — og lad det lære kernerne selv

Nu samler vi klodserne. Et klassisk lille CNN ser sådan ud:

```
billede (1×28×28)
  → Conv2d(1→16 kerner, 3×3, padding=1) → ReLU → MaxPool(2)     "find 16 slags småmønstre, zoom ud"
  → Conv2d(16→32 kerner, 3×3, padding=1) → ReLU → MaxPool(2)    "kombinér til 32 større mønstre, zoom ud"
  → flad ud → Linear(32·7·7 → 10)                               "afgør: hvilken tøjtype?"
```

Bemærk logikken: konvolutionslagene finder mønstre (kanter → hjørner → ærmer → sko-form),
poolingen zoomer gradvist ud, og til allersidst samler ét dense-lag det hele til en
beslutning. Størrelserne undervejs: 28 → 28 → 14 → 14 → 7 (padding=1 bevarer størrelsen,
poolingen halverer — regn selv efter med formlen fra 19.7!).

Og træningsloopet? **Fuldstændig uændret** fra Intro-ML — CrossEntropyLoss, Adam,
mini-batches. Et CNN er bare et `nn.Module` med andre lag indeni:

In [ ]:
class ToejNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)    # 1 kanal ind → 16 kerner
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)   # 16 ind → 32 kerner
        self.pool = nn.MaxPool2d(2)
        self.aktivering = nn.ReLU()
        self.fc = nn.Linear(32 * 7 * 7, 10)              # 32 kort á 7×7 → 10 klasser

    def forward(self, x):
        x = self.pool(self.aktivering(self.conv1(x)))    # 28 → 28 → 14
        x = self.pool(self.aktivering(self.conv2(x)))    # 14 → 14 → 7
        x = x.reshape(x.shape[0], -1)                    # flad ud: (N, 1568)
        return self.fc(x)                                # rå point (CrossEntropy-reglen!)

model = ToejNet()
print(model)
print("parametre:", sum(p.numel() for p in model.parameters()))

In [ ]:
import time

torch.manual_seed(42)
model = ToejNet()
tabsfunktion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

start = time.time()
for epoke in range(5):
    for i in range(0, len(X_traen), 64):
        optimizer.zero_grad()
        tab = tabsfunktion(model(X_traen[i:i + 64]), y_traen[i:i + 64])
        tab.backward()
        optimizer.step()
    print(f"epoke {epoke + 1}: tab på sidste portion = {tab.item():.4f}")
print(f"træningstid: {time.time() - start:.0f} s")

In [ ]:
with torch.no_grad():
    gaet = model(X_test).argmax(dim=1)
print(f"CNN test-accuracy: {(gaet == y_test).float().mean().item():.1%}")

## Duellen: CNN mod dense-netværk

Er al konvolutions-besværet det værd? Lad os træne jeres velkendte dense-net fra
Intro-ML på præcis samme data — og sammenligne både accuracy OG antal parametre:

In [ ]:
torch.manual_seed(42)
dense_model = nn.Sequential(nn.Flatten(),                 # (N,1,28,28) → (N,784)
                            nn.Linear(784, 128), nn.ReLU(),
                            nn.Linear(128, 10))
optimizer = torch.optim.Adam(dense_model.parameters(), lr=0.001)

for epoke in range(5):
    for i in range(0, len(X_traen), 64):
        optimizer.zero_grad()
        tab = tabsfunktion(dense_model(X_traen[i:i + 64]), y_traen[i:i + 64])
        tab.backward()
        optimizer.step()

with torch.no_grad():
    dense_gaet = dense_model(X_test).argmax(dim=1)

print(f"CNN:   {(gaet == y_test).float().mean().item():.1%}  med {sum(p.numel() for p in model.parameters()):>7} parametre")
print(f"dense: {(dense_gaet == y_test).float().mean().item():.1%}  med {sum(p.numel() for p in dense_model.parameters()):>7} parametre")

CNN'et vinder — med ca. **5× færre parametre**. Struktur-viden slår rå størrelse.
(Og forspringet vokser med mere data og træning: på det fulde FashionMNIST når denne
arkitektur ~90 %+, og på rigtige fotos er dense-net helt chanceløse.)

## Hvor tager modellen fejl? Forvirringsmatricen

Accuracy er ét tal — **forvirringsmatricen** viser HVILKE klasser der forveksles:

In [ ]:
plot_forvirringsmatrix(y_test, gaet, klassenavne=klassenavne)

Kig på de store tal UDEN FOR diagonalen: skjorte ↔ T-shirt ↔ frakke ↔ sweater
forveksles flittigt (fair nok — i 28×28 gråtoner!), mens bukser og tasker næsten aldrig
rammes forkert.

## Kig ind i maskinen: de lærte kerner

Vi håndbyggede kant-kerner i afsnit 5 — hvad har NETVÆRKET selv fundet på? `conv1` har
16 kerner à 3×3, og vi kan også se **feature maps**: hvad hver kerne "ser" i et
konkret billede:

In [ ]:
kerner = model.conv1.weight.detach()          # shape (16, 1, 3, 3)

fig, akser = plt.subplots(2, 8, figsize=(12, 3.5))
for i, akse in enumerate(akser.ravel()):
    akse.imshow(kerner[i, 0], cmap="gray")
    akse.set_title(f"kerne {i}", fontsize=8)
    akse.axis("off")
plt.suptitle("De 16 lærte kerner i conv1")
plt.show()

In [ ]:
with torch.no_grad():
    feature_maps = model.aktivering(model.conv1(sko.reshape(1, 1, 28, 28)))

fig, akser = plt.subplots(2, 8, figsize=(12, 3.5))
for i, akse in enumerate(akser.ravel()):
    akse.imshow(feature_maps[0, i], cmap="gray")
    akse.axis("off")
plt.suptitle("Hvad de 16 kerner ser i sneakeren (feature maps)")
plt.show()

### Opgaver

##### Opgave 6.1
Brug `vis_billeder` til at se 10 billeder, som CNN'et gætter FORKERT (mønstret kender I
fra Intro-ML: `(gaet!= y_test).nonzero()`). Passer fejlene med forvirringsmatricens
værste par — og er der nogen, hvor I selv ville have gættet det samme som modellen?

In [ ]:
forkerte = (gaet != y_test).nonzero().squeeze()
print("antal fejl:", len(forkerte))

vis_billeder(X_test[forkerte[:10]], y_test[forkerte[:10]],
             forudsigelser=gaet[forkerte[:10]], n=10, klassenavne=klassenavne)

##### Opgave 6.2
Skru op for antallet af kerner: prøv (32, 64) i stedet for (16, 32) — husk at `fc`-lagets
input så også ændrer sig! Hvad sker der med accuracy og træningstid?

In [ ]:
class StortToejNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)    # ← 16 → 32
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)   # ← 16, 32 → 32, 64
        self.pool = nn.MaxPool2d(2)
        self.aktivering = nn.ReLU()
        self.fc = nn.Linear(32 * 7 * 7, 10)              # ← hvad skal 32 være nu?

    def forward(self, x):
        x = self.pool(self.aktivering(self.conv1(x)))
        x = self.pool(self.aktivering(self.conv2(x)))
        x = x.reshape(x.shape[0], -1)
        return self.fc(x)

torch.manual_seed(42)
stort = StortToejNet()
optimizer = torch.optim.Adam(stort.parameters(), lr=0.001)
start = time.time()
for epoke in range(5):
    for i in range(0, len(X_traen), 64):
        optimizer.zero_grad()
        tab = tabsfunktion(stort(X_traen[i:i + 64]), y_traen[i:i + 64])
        tab.backward()
        optimizer.step()
with torch.no_grad():
    acc = (stort(X_test).argmax(dim=1) == y_test).float().mean()
print(f"accuracy {acc.item():.1%} på {time.time() - start:.0f} s")

##### Opgave 6.3
Udfyld parametertællingerne og beregn, hvor mange gange flere parametre dense-nettet
har. Hvor sidder langt de fleste af CNN'ets parametre — i konvolutionslagene eller i
`fc`-laget?

In [ ]:
cnn_total = sum(p.numel() for p in model.parameters())
dense_total = ...

print(f"CNN: {cnn_total}, dense: {dense_total}, forhold: {dense_total / cnn_total:.1f}x")

# og fordelt på lag i CNN'et:
for navn, p in model.named_parameters():
    print(f"{navn:15s} {p.numel():>6} tal")

##### Opgave 6.4 (find fejlen)
Nogen har slettet en linje i `forward` — og nu kaster netværket en lang shape-fejl, når
man kalder det. Kør cellen, læs fejlen, og sæt den manglende linje ind igen. Hvorfor kan
`fc`-laget ikke bare spise conv-outputtet direkte?

In [ ]:
class OedelagtNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.aktivering = nn.ReLU()
        self.fc = nn.Linear(32 * 7 * 7, 10)

    def forward(self, x):
        x = self.pool(self.aktivering(self.conv1(x)))
        x = self.pool(self.aktivering(self.conv2(x)))
        return self.fc(x)

model_o = OedelagtNet()
print(model_o(X_test[:4]))

##### Opgave 6.5
Tilføj et TREDJE konvolutionslag (32 → 64 kerner, UDEN padding) efter conv2 — og uden
pooling efter (7×7 er allerede småt). Brug formlen fra 19.7 til at regne ud, hvad
`fc`-lagets input skal være, FØR I kører. Blev accuracy bedre?

In [ ]:
class DybtNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3)     # ingen padding!
        self.pool = nn.MaxPool2d(2)
        self.aktivering = nn.ReLU()
        self.fc = nn.Linear(..., 10)          # ← 7×7-kortene går gennem conv3 — hvad så?

    def forward(self, x):
        x = self.pool(self.aktivering(self.conv1(x)))
        x = self.pool(self.aktivering(self.conv2(x)))
        x = self.aktivering(self.conv3(x))
        x = x.reshape(x.shape[0], -1)
        return self.fc(x)

torch.manual_seed(42)
dybt = DybtNet()
optimizer = torch.optim.Adam(dybt.parameters(), lr=0.001)
for epoke in range(5):
    for i in range(0, len(X_traen), 64):
        optimizer.zero_grad()
        tab = tabsfunktion(dybt(X_traen[i:i + 64]), y_traen[i:i + 64])
        tab.backward()
        optimizer.step()
with torch.no_grad():
    acc = (dybt(X_test).argmax(dim=1) == y_test).float().mean()
print(f"accuracy: {acc.item():.1%}")

##### Opgave 6.6
Robusthedstesten fra opgave 5.9: forskyd ALLE testbilleder 3 pixels til højre med
`torch.roll` og mål både CNN'ets og dense-nettets accuracy på de forskudte billeder.
Hvem tåler flytningen bedst — og hvorfor er ingen af dem uberørt?

In [ ]:
X_test_forskudt = torch.roll(X_test, shifts=3, dims=3)   # 3 pixels mod højre

vis_billeder(X_test_forskudt, y_test, n=5, klassenavne=klassenavne)

with torch.no_grad():
    cnn_acc = (model(X_test_forskudt).argmax(dim=1) == y_test).float().mean()
    dense_acc = ...
print(f"forskudt: CNN {cnn_acc.item():.1%} | dense {...}")

##### Opgave 6.7
Udfyld softmax-kaldet og find det billede i testsættet, hvor modellen er MEST i tvivl
(mindste max-sandsynlighed). Vis billedet og dets sandsynligheds-søjler — hvad er
modellen i tvivl imellem?

In [ ]:
with torch.no_grad():
    sandsynligheder = torch.softmax(model(X_test), dim=...)

max_sandsynlighed = sandsynligheder.max(dim=1).values
i = max_sandsynlighed.argmin().item()          # mest usikre billede

fig, akser = plt.subplots(1, 2, figsize=(10, 3.5))
akser[0].imshow(X_test[i].squeeze(), cmap="gray")
akser[0].set_title(f"label: {klassenavne[y_test[i]]}")
akser[0].axis("off")
akser[1].bar(klassenavne, sandsynligheder[i])
akser[1].tick_params(axis="x", rotation=45)
plt.show()

##### Opgave 6.8
Feature maps-figuren viser, hvad hver kerne "ser". Kør den med en **taske** og en
**støvle** i stedet for sneakeren. Kan I finde en kerne, der tydeligt er "kant-agtig"
(sammenlign med jeres håndbyggede fra afsnit 5)? Og en, der mest ligner støj?

In [ ]:
billede = X_traen[y_traen == 8][0]      # taske — prøv også 9 (støvle)

with torch.no_grad():
    fm = model.aktivering(model.conv1(billede.reshape(1, 1, 28, 28)))

fig, akser = plt.subplots(2, 8, figsize=(12, 3.5))
for i, akse in enumerate(akser.ravel()):
    akse.imshow(fm[0, i], cmap="gray")
    akse.set_title(f"kerne {i}", fontsize=8)
    akse.axis("off")
plt.show()

##### Opgave 6.9 (ekstra)
**CIFAR-10: rigtige FARVEBILLEDER.** Koden nedenfor er komplet: den henter CIFAR-10
(fly, biler, katte... — ~170 MB, men Kaggle er hurtig), viser billederne og træner
et farve-CNN (3 input-kanaler!). Kør den, og eksperimentér så: flere epoker? flere
kerner? Hvor langt kan I komme over de ~50 %? (Tilfældig gætning er 10 % — men perfekt
er UMULIGT på 3 epoker og 4.000 billeder.)

In [ ]:
import torchvision

# CIFAR-10 hentes fra Kaggle (hurtigt!) — torchvision læser de udpakkede filer:
sti_cifar = kagglehub.dataset_download("pankrzysiu/cifar10-python")
cifar_traen = torchvision.datasets.CIFAR10(root=sti_cifar, train=True)
cifar_test = torchvision.datasets.CIFAR10(root=sti_cifar, train=False)

#  Plan B (hvis Kaggle driller) — hent direkte fra kilden (kan være LANGSOM):
# cifar_traen = torchvision.datasets.CIFAR10(root="cifar_data", train=True, download=True)
# cifar_test = torchvision.datasets.CIFAR10(root="cifar_data", train=False, download=True)

X_c = torch.tensor(cifar_traen.data[:4000] / 255.0, dtype=torch.float32).permute(0, 3, 1, 2)
y_c = torch.tensor(cifar_traen.targets[:4000], dtype=torch.long)
X_ct = torch.tensor(cifar_test.data[:1000] / 255.0, dtype=torch.float32).permute(0, 3, 1, 2)
y_ct = torch.tensor(cifar_test.targets[:1000], dtype=torch.long)
cifar_navne = ["fly", "bil", "fugl", "kat", "hjort", "hund", "frø", "hest", "skib", "lastbil"]
print("X_c:", X_c.shape, " ← 3 kanaler: rød, grøn, blå!")

fig, akser = plt.subplots(1, 6, figsize=(12, 2.5))
for i, akse in enumerate(akser):
    akse.imshow(X_c[i].permute(1, 2, 0))       # tilbage til (H, B, kanaler) for imshow
    akse.set_title(cifar_navne[y_c[i]])
    akse.axis("off")
plt.show()

class FarveNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)   # 3 kanaler ind!
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.aktivering = nn.ReLU()
        self.fc = nn.Linear(32 * 8 * 8, 10)                       # 32 → 16 → 8 med padding=1

    def forward(self, x):
        x = self.pool(self.aktivering(self.conv1(x)))
        x = self.pool(self.aktivering(self.conv2(x)))
        x = x.reshape(x.shape[0], -1)
        return self.fc(x)

torch.manual_seed(42)
farve_model = FarveNet()
optimizer = torch.optim.Adam(farve_model.parameters(), lr=0.001)
tabsfunktion = nn.CrossEntropyLoss()
for epoke in range(3):
    for i in range(0, len(X_c), 64):
        optimizer.zero_grad()
        tab = tabsfunktion(farve_model(X_c[i:i + 64]), y_c[i:i + 64])
        tab.backward()
        optimizer.step()
    print(f"epoke {epoke + 1} færdig")

with torch.no_grad():
    acc = (farve_model(X_ct).argmax(dim=1) == y_ct).float().mean()
print(f"CIFAR-10 accuracy: {acc.item():.1%}  (tilfældigt = 10 %)")

##### Opgave 6.10 (ekstra)
**Det ondeste eksperiment:** omrokér alle 784 pixels med den SAMME tilfældige
permutation i alle billeder (så informationen bevares, men naboskabet ødelægges), og
træn både CNN og dense-net på de omrokerede billeder. Forudsig FØRST: hvem rammes
hårdest — og hvorfor? Kør så og se.

In [ ]:
perm = torch.randperm(784)

def omroker(X):
    return X.reshape(-1, 784)[:, perm].reshape(-1, 1, 28, 28)

vis_billeder(omroker(X_traen), y_traen, n=5, klassenavne=klassenavne)   # "tøj"...

# træn nu et frisk ToejNet og et frisk dense-net på omroker(X_traen),
# og mål på omroker(X_test) — genbrug træningscellerne fra tidligere
...

# Godt gået!

I kan nu: regne konvolutioner i hånden, bygge kant-detektorer, læse og bygge
CNN-arkitekturer (med formlen for output-størrelser), sammenligne CNN og dense fair —
og I har set med egne øjne, at CNN'ets kraft kommer fra naboskabs-antagelsen.

**Værktøjskassen:** tabeldata → træer/skove · billeder → **CNN** · sekvenser → RNN
(næste notebook!) · og på campens sidste dag: sprog → transformere.